# CS-781 Phase 1 — Architecture 2 (encoder–decoder), v2 training

This notebook trains my encoder–decoder model on the data shards and the ByteLevel BPE tokenizer built in `archit-2-0`, and generates the Phase-1 continuations.

**Model (v2 preset).** 6 encoder + 8 decoder layers, d_model 768, 12 heads, d_ff 3072, learned positions, pre-LayerNorm, tied embeddings, 143.7M parameters. (The `v1` preset is my earlier 61M model.)

**Decoding.** Repetition control (n-gram blocking + repetition penalty) and forced 128-word length. The decoding setting is selected by BERTScore on held-out prompts. Plain greedy decoding loops and scores much lower.

**Output.** `submission.zip` with `prediction.json` for the 1000 test prompts.


In [1]:
# ============================================================
# CELL 1: SETTINGS  (the only cell I normally edit)
# ============================================================
import os, glob, json, math, time, random, zipfile, shutil
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass

PRESET = "v2"        # "v2" = new ~143M model | "v1" = my old 61M model (use with TRAIN=False)
TRAIN  = True        # False = no training, load weights and only decode
SMOKE  = False        # True = quick end-to-end bug check. Set False for the real run.
TIME_BUDGET_HOURS = 5.5       # training wall-clock budget (LR schedule is tied to it)
WEIGHTS_PATH = None           # only for TRAIN=False. None = auto-search for best_model.pt
TEST_JSON_PATH = None         # None = download official test.json from Hugging Face (needs Kaggle secret "HF")

PRESETS = {
    "v2": dict(d_model=768, heads=12, enc_layers=6, dec_layers=8, d_ff=3072, dropout=0.05, label_smoothing=0.0),
    "v1": dict(d_model=512, heads=8,  enc_layers=6, dec_layers=6, d_ff=2048, dropout=0.10, label_smoothing=0.1),
}

# ---- training hyper-parameters (v2)
BATCH_PER_FORWARD = 16     # total over both GPUs (8 per T4). If CUDA OOM -> 8
ACCUM             = 4      # effective batch = 16 * 4 = 64 pairs
PEAK_LR           = 3e-4
MIN_LR_RATIO      = 0.05
WARMUP_STEPS      = 400
WEIGHT_DECAY      = 0.01
MAX_GRAD_NORM     = 1.0
EVAL_EVERY        = 500    # optimizer steps
VAL_SUBSET        = 3000   # validation pairs used during training
LOG_EVERY         = 50
SWEEP_N           = 200    # validation prompts used to choose the decoding setting
TEST_LIMIT        = None   # None = all test prompts

if SMOKE:
    TIME_BUDGET_HOURS = 0.03; ACCUM = 1; WARMUP_STEPS = 5; EVAL_EVERY = 15; VAL_SUBSET = 128
    LOG_EVERY = 5; SWEEP_N = 16; TEST_LIMIT = 16

SEED = 1234
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N_GPU  = torch.cuda.device_count()
OUT_DIR = "/kaggle/working/arch2_v2_smoke" if SMOKE else "/kaggle/working/arch2_v2_training"
os.makedirs(OUT_DIR, exist_ok=True)

print("PRESET:", PRESET, "| TRAIN:", TRAIN, "| SMOKE:", SMOKE)
print("GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
print("Time budget (h):", TIME_BUDGET_HOURS)

PRESET: v2 | TRAIN: True | SMOKE: False
GPUs: 2 ['Tesla T4', 'Tesla T4']
Time budget (h): 5.5


In [2]:
# ============================================================
# CELL 2: LOCATE DATA + TOKENIZER, LOAD INTO RAM
# ============================================================
hits  = glob.glob("/kaggle/input/**/arch2_final_processed/preprocessing_state.json", recursive=True)
hits += glob.glob("/kaggle/working/**/arch2_final_processed/preprocessing_state.json", recursive=True)
assert hits, "arch2_final_processed not found. Attach the output of notebook archit-2-0 as an input."
DATA_DIR = os.path.dirname(hits[0])
TOK_DIR  = os.path.join(DATA_DIR, "tokenizer")
train_files = sorted(glob.glob(f"{DATA_DIR}/train/**/*.npz", recursive=True))
val_files   = sorted(glob.glob(f"{DATA_DIR}/validation/**/*.npz", recursive=True))
print("DATA_DIR:", DATA_DIR)
print("train shards:", len(train_files), "| validation shards:", len(val_files))
assert len(val_files) > 0 and (len(train_files) > 0 or not TRAIN)

def load_split(files):
    enc, dec, lab = [], [], []
    for p in files:
        with np.load(p) as z:
            enc.append(z["encoder_input_ids"].astype(np.int16))
            dec.append(z["decoder_input_ids"].astype(np.int16))
            lab.append(z["labels"].astype(np.int16))
    return np.concatenate(enc), np.concatenate(dec), np.concatenate(lab)

t0 = time.time()
va_enc, va_dec, va_lab = load_split(val_files)
if TRAIN:
    tr_enc, tr_dec, tr_lab = load_split(train_files)
    print(f"train pairs {len(tr_enc):,} | RAM {(tr_enc.nbytes+tr_dec.nbytes+tr_lab.nbytes)/2**30:.2f} GB")
print(f"val pairs {len(va_enc):,} | loaded in {time.time()-t0:.0f}s")

from tokenizers import ByteLevelBPETokenizer
tokenizer = ByteLevelBPETokenizer(os.path.join(TOK_DIR, "vocab.json"), os.path.join(TOK_DIR, "merges.txt"))
assert tokenizer.get_vocab_size() == 32000
assert [tokenizer.token_to_id(t) for t in ("<pad>", "<s>", "</s>", "<unk>")] == [0, 1, 2, 3]
print("tokenizer OK, vocab", tokenizer.get_vocab_size())

DATA_DIR: /kaggle/input/notebooks/dummy595/archit-2-0/arch2_final_processed
train shards: 27 | validation shards: 6
train pairs 250,859 | RAM 0.72 GB
val pairs 26,364 | loaded in 8s
tokenizer OK, vocab 32000


In [3]:
# ============================================================
# CELL 3: MODEL  (encoder-decoder Transformer, written from scratch)
#   - same module/parameter names as my v1 model, so old weights still load
#   - NEW: key/value cache for fast generation, scaled residual init, no pad-mask in decoder training
# ============================================================
@dataclass
class Cfg:
    vocab_size: int = 32000
    enc_max: int = 1024
    dec_max: int = 256
    pad: int = 0
    bos: int = 1
    eos: int = 2
    unk: int = 3
    d_model: int = 768
    heads: int = 12
    enc_layers: int = 6
    dec_layers: int = 8
    d_ff: int = 3072
    dropout: float = 0.05
    label_smoothing: float = 0.0

cfg = Cfg(**PRESETS[PRESET])

class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_length, pad_id, dropout):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)
        nn.init.normal_(self.token_embedding.weight, 0.0, d_model ** -0.5)
        nn.init.normal_(self.position_embedding.weight, 0.0, 0.02)
        with torch.no_grad():
            self.token_embedding.weight[pad_id].zero_()
    def forward(self, ids, offset=0):
        T = ids.size(1)
        pos = torch.arange(offset, offset + T, device=ids.device).unsqueeze(0)
        x = self.token_embedding(ids) * math.sqrt(self.d_model) + self.position_embedding(pos)
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model, self.h, self.dh, self.p = d_model, num_heads, d_model // num_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        for l in (self.q_proj, self.k_proj, self.v_proj, self.out_proj):
            nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.dh).transpose(1, 2)
    def _merge(self, x):
        B, _, T, _ = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, self.d_model)
    def project_kv(self, kv):
        return self._split(self.k_proj(kv)), self._split(self.v_proj(kv))
    def attend(self, x, k, v, mask=None, causal=False):
        q = self._split(self.q_proj(x))
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask,
            dropout_p=self.p if self.training else 0.0, is_causal=causal)
        return self.out_proj(self._merge(out))
    def forward(self, query, key_value=None, key_padding_mask=None, causal=False):
        kv = query if key_value is None else key_value
        k, v = self.project_kv(kv)
        mask = None
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :].bool()
            if causal:
                cm = torch.tril(torch.ones(query.size(1), kv.size(1), dtype=torch.bool, device=query.device))[None, None]
                mask = mask & cm
                causal = False
        return self.attend(query, k, v, mask=mask, causal=causal)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)
        for l in (self.fc1, self.fc2):
            nn.init.xavier_uniform_(l.weight); nn.init.zeros_(l.bias)
    def forward(self, x):
        return self.fc2(self.dropout(self.activation(self.fc1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, heads, dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x, src_mask):
        x = x + self.dropout1(self.self_attn(self.norm1(x), key_padding_mask=src_mask))
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, heads, dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, heads, dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout3 = nn.Dropout(dropout)
    def forward(self, x, enc_out, src_mask):
        x = x + self.dropout1(self.self_attn(self.norm1(x), causal=True))
        x = x + self.dropout2(self.cross_attn(self.norm2(x), key_value=enc_out, key_padding_mask=src_mask))
        x = x + self.dropout3(self.ffn(self.norm3(x)))
        return x
    def step(self, x, cache, enc_k, enc_v, src_mask):
        """One incremental decoding step with a self-attention key/value cache."""
        h = self.norm1(x)
        k_new, v_new = self.self_attn.project_kv(h)
        if cache is None:
            k, v = k_new, v_new
        else:
            k, v = torch.cat([cache[0], k_new], dim=2), torch.cat([cache[1], v_new], dim=2)
        x = x + self.self_attn.attend(h, k, v)
        x = x + self.cross_attn.attend(self.norm2(x), enc_k, enc_v, mask=src_mask[:, None, None, :])
        x = x + self.ffn(self.norm3(x))
        return x, (k, v)

class Encoder(nn.Module):
    def __init__(self, embedding, c):
        super().__init__()
        self.pad_token_id = c.pad
        self.embedding = embedding
        self.layers = nn.ModuleList([EncoderLayer(c.d_model, c.heads, c.d_ff, c.dropout) for _ in range(c.enc_layers)])
        self.final_norm = nn.LayerNorm(c.d_model)
    def forward(self, ids, src_mask):
        x = self.embedding(ids)
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.final_norm(x)

class Decoder(nn.Module):
    def __init__(self, embedding, c):
        super().__init__()
        self.pad_token_id = c.pad
        self.embedding = embedding
        self.layers = nn.ModuleList([DecoderLayer(c.d_model, c.heads, c.d_ff, c.dropout) for _ in range(c.dec_layers)])
        self.final_norm = nn.LayerNorm(c.d_model)
    def forward(self, ids, enc_out, src_mask):
        x = self.embedding(ids)
        for layer in self.layers:
            x = layer(x, enc_out, src_mask)
        return self.final_norm(x)

class EncoderDecoderLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        enc_emb = TransformerEmbedding(c.vocab_size, c.d_model, c.enc_max, c.pad, c.dropout)
        dec_emb = TransformerEmbedding(c.vocab_size, c.d_model, c.dec_max, c.pad, c.dropout)
        dec_emb.token_embedding.weight = enc_emb.token_embedding.weight          # shared embeddings
        self.encoder = Encoder(enc_emb, c)
        self.decoder = Decoder(dec_emb, c)
        self.lm_head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.lm_head.weight = dec_emb.token_embedding.weight                     # tied output layer
        with torch.no_grad():                                                    # scaled residual init (stability)
            se, sd = 1 / math.sqrt(2 * c.enc_layers), 1 / math.sqrt(3 * c.dec_layers)
            for l in self.encoder.layers:
                l.self_attn.out_proj.weight.mul_(se); l.ffn.fc2.weight.mul_(se)
            for l in self.decoder.layers:
                l.self_attn.out_proj.weight.mul_(sd); l.cross_attn.out_proj.weight.mul_(sd); l.ffn.fc2.weight.mul_(sd)

    def forward(self, src_ids, dec_ids, labels):
        """Training / validation loss. Returns a (1,) tensor so DataParallel can gather it."""
        src_mask = src_ids != self.c.pad
        enc = self.encoder(src_ids, src_mask)
        hid = self.decoder(dec_ids, enc, src_mask)          # causal only; padding sits at the end and is ignored by labels
        valid = labels != -100
        logits = self.lm_head(hid[valid])
        ls = self.c.label_smoothing if self.training else 0.0
        return F.cross_entropy(logits.float(), labels[valid], label_smoothing=ls).unsqueeze(0)

    @torch.no_grad()
    def start_generation(self, src_ids):
        src_mask = src_ids != self.c.pad
        enc = self.encoder(src_ids, src_mask)
        enc_kv = [l.cross_attn.project_kv(enc) for l in self.decoder.layers]
        return src_mask, enc_kv

    @torch.no_grad()
    def step(self, tok, pos, caches, enc_kv, src_mask):
        x = self.decoder.embedding(tok, offset=pos)
        new_caches = []
        for i, layer in enumerate(self.decoder.layers):
            x, c = layer.step(x, caches[i], enc_kv[i][0], enc_kv[i][1], src_mask)
            new_caches.append(c)
        x = self.decoder.final_norm(x)
        return self.lm_head(x[:, -1, :]).float(), new_caches

model = EncoderDecoderLM(cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
ptrs = {model.encoder.embedding.token_embedding.weight.data_ptr(),
        model.decoder.embedding.token_embedding.weight.data_ptr(), model.lm_head.weight.data_ptr()}
print(f"Parameters: {n_params:,} ({n_params/1e6:.2f}M) | tied embeddings: {len(ptrs)==1}")
assert 50e6 <= n_params <= 300e6, "parameter count outside the 50M-300M rule"
if PRESET == "v1":
    assert n_params == 61_179_904

Parameters: 143,703,552 (143.70M) | tied embeddings: True


In [4]:
# ============================================================
# CELL 4: TRAINING  (time-budgeted, resumable, true-NLL validation)
# ============================================================
class InMemoryLoader:
    def __init__(self, enc, dec, lab, bs, shuffle, seed=42):
        self.enc, self.dec, self.lab, self.bs, self.shuffle, self.seed = enc, dec, lab, bs, shuffle, seed
        self.n = len(enc)
    def num_batches(self):
        return (self.n + self.bs - 1) // self.bs
    def batches(self, epoch=0, start_batch=0):
        order = np.random.default_rng(self.seed + epoch).permutation(self.n) if self.shuffle else np.arange(self.n)
        for bi in range(start_batch, self.num_batches()):
            idx = order[bi * self.bs:(bi + 1) * self.bs]
            e, d, l = self.enc[idx], self.dec[idx], self.lab[idx]
            e_len = max(1, int((e != 0).sum(1).max()))
            t_len = max(1, int((l != -100).sum(1).max()))
            yield bi, {"src": torch.from_numpy(e[:, :e_len].astype(np.int64)),
                       "dec": torch.from_numpy(d[:, :t_len].astype(np.int64)),
                       "lab": torch.from_numpy(l[:, :t_len].astype(np.int64))}

LATEST = os.path.join(OUT_DIR, "latest.pt")
BEST   = os.path.join(OUT_DIR, "best_model.pt")
HIST   = os.path.join(OUT_DIR, "history.json")

def run_training():
    sub = np.linspace(0, len(va_enc) - 1, min(VAL_SUBSET, len(va_enc))).astype(int)
    val_loader = InMemoryLoader(va_enc[sub], va_dec[sub], va_lab[sub], 32, shuffle=False)
    train_loader = InMemoryLoader(tr_enc, tr_dec, tr_lab, BATCH_PER_FORWARD, shuffle=True, seed=SEED)
    dp = nn.DataParallel(model, device_ids=list(range(N_GPU))) if N_GPU > 1 else model

    decay, no_decay = [], []
    for name, p in model.named_parameters():
        (no_decay if (p.ndim < 2 or "embedding" in name or "lm_head" in name) else decay).append(p)
    optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                                   {"params": no_decay, "weight_decay": 0.0}],
                                  lr=PEAK_LR, betas=(0.9, 0.98), eps=1e-8, fused=torch.cuda.is_available())
    scaler = torch.amp.GradScaler("cuda")

    state = dict(step=0, epoch=0, next_batch=0, elapsed=0.0, best_val=float("inf"), history=[])
    resume = glob.glob("/kaggle/input/**/arch2_v2_training/latest.pt", recursive=True)
    resume = LATEST if os.path.exists(LATEST) else (resume[0] if resume else None)
    if resume and not SMOKE:
        ck = torch.load(resume, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"]); scaler.load_state_dict(ck["scaler"])
        state = ck["state"]
        print("RESUMED from", resume, "| step", state["step"], "| elapsed h", state["elapsed"] / 3600)

    budget = TIME_BUDGET_HOURS * 3600
    t_sess, base_el = time.time(), state["elapsed"]
    elapsed = lambda: base_el + (time.time() - t_sess)

    def lr_at(step, frac):
        warm = min(1.0, (step + 1) / WARMUP_STEPS)
        cos = MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, frac)))
        return PEAK_LR * warm * cos

    @torch.no_grad()
    def validate():
        dp.eval(); tot, cnt = 0.0, 0
        for _, b in val_loader.batches(0):
            with torch.autocast("cuda", dtype=torch.float16):
                loss = dp(b["src"].to(DEVICE), b["dec"].to(DEVICE), b["lab"].to(DEVICE)).mean()
            n = int((b["lab"] != -100).sum()); tot += loss.item() * n; cnt += n
        dp.train()
        return tot / cnt

    def save_all(val_nll):
        tmp = LATEST + ".tmp"
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "state": state}, tmp)
        os.replace(tmp, LATEST)
        if val_nll < state["best_val"]:
            state["best_val"] = val_nll
            torch.save(model.state_dict(), BEST + ".tmp"); os.replace(BEST + ".tmp", BEST)
            print(f"   new best val NLL {val_nll:.4f} -> best_model.pt")
        json.dump(state["history"], open(HIST, "w"), indent=1)

    dp.train(); optimizer.zero_grad(set_to_none=True)
    accum, run_loss, run_n, t_log, done = 0, 0.0, 0, time.time(), False
    print(f"batches/epoch {train_loader.num_batches():,} | optimizer steps/epoch ~{train_loader.num_batches()//ACCUM:,}")
    while not done:
        for bi, b in train_loader.batches(state["epoch"], state["next_batch"]):
            with torch.autocast("cuda", dtype=torch.float16):
                loss = dp(b["src"].to(DEVICE), b["dec"].to(DEVICE), b["lab"].to(DEVICE)).mean()
            state["next_batch"] = bi + 1
            if not torch.isfinite(loss):
                print("[warn] non-finite loss, skipping"); optimizer.zero_grad(set_to_none=True); accum = 0; continue
            scaler.scale(loss / ACCUM).backward()
            accum += 1; run_loss += loss.item(); run_n += 1
            if accum < ACCUM:
                continue
            frac = elapsed() / budget
            lr = lr_at(state["step"], frac)
            for g in optimizer.param_groups: g["lr"] = lr
            scaler.unscale_(optimizer)
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            accum = 0; state["step"] += 1
            if state["step"] % LOG_EVERY == 0:
                dt = time.time() - t_log; t_log = time.time()
                print(f"ep {state['epoch']+1} | step {state['step']} | loss {run_loss/run_n:.4f} | gn {float(gn):.2f} | "
                      f"lr {lr:.2e} | {LOG_EVERY*ACCUM*BATCH_PER_FORWARD/dt:.0f} pairs/s | "
                      f"{elapsed()/3600:.2f}h / {TIME_BUDGET_HOURS}h")
                run_loss, run_n = 0.0, 0
            end = frac >= 1.0
            if state["step"] % EVAL_EVERY == 0 or end:
                v = validate(); state["elapsed"] = elapsed()
                state["history"].append({"step": state["step"], "epoch": state["epoch"] + state["next_batch"] / train_loader.num_batches(),
                                         "elapsed_h": state["elapsed"] / 3600, "val_nll": v, "val_ppl": math.exp(min(v, 20))})
                print(f"== VAL step {state['step']} | NLL {v:.4f} | PPL {math.exp(min(v,20)):.2f}")
                save_all(v)
            if end:
                done = True; break
        else:
            state["epoch"] += 1; state["next_batch"] = 0
    return state

if TRAIN:
    train_state = run_training()
    print("Training finished. Best val NLL:", train_state["best_val"])

batches/epoch 15,679 | optimizer steps/epoch ~3,919
ep 1 | step 50 | loss 13.7482 | gn 1.25 | lr 3.75e-05 | 39 pairs/s | 0.02h / 5.5h
ep 1 | step 100 | loss 7.3515 | gn 0.78 | lr 7.50e-05 | 39 pairs/s | 0.05h / 5.5h
ep 1 | step 150 | loss 6.4365 | gn 0.80 | lr 1.12e-04 | 39 pairs/s | 0.07h / 5.5h
ep 1 | step 200 | loss 5.9232 | gn 1.00 | lr 1.50e-04 | 39 pairs/s | 0.09h / 5.5h
ep 1 | step 250 | loss 5.6255 | gn 0.74 | lr 1.87e-04 | 38 pairs/s | 0.12h / 5.5h
ep 1 | step 300 | loss 5.3757 | gn 0.69 | lr 2.25e-04 | 38 pairs/s | 0.14h / 5.5h
ep 1 | step 350 | loss 5.2326 | gn 0.88 | lr 2.62e-04 | 38 pairs/s | 0.16h / 5.5h
ep 1 | step 400 | loss 5.0688 | gn 0.68 | lr 2.99e-04 | 38 pairs/s | 0.18h / 5.5h
ep 1 | step 450 | loss 4.9486 | gn 0.73 | lr 2.99e-04 | 38 pairs/s | 0.21h / 5.5h
ep 1 | step 500 | loss 4.8101 | gn 0.62 | lr 2.99e-04 | 38 pairs/s | 0.23h / 5.5h
== VAL step 500 | NLL 5.0588 | PPL 157.40
   new best val NLL 5.0588 -> best_model.pt
ep 1 | step 550 | loss 4.6787 | gn 1.07 | 

In [5]:
# ============================================================
# CELL 5: LOAD THE BEST WEIGHTS FOR DECODING
# ============================================================
wp = WEIGHTS_PATH
if wp is None:
    cand = [BEST] if os.path.exists(BEST) else glob.glob("/kaggle/input/**/best_model.pt", recursive=True)
    assert cand, "best_model.pt not found. Set WEIGHTS_PATH."
    wp = cand[0]
sd = torch.load(wp, map_location=DEVICE, weights_only=False)
print("loading:", wp, "|", model.load_state_dict(sd, strict=True))
model.eval()

loading: /kaggle/working/arch2_v2_training/best_model.pt | <All keys matched successfully>


EncoderDecoderLM(
  (encoder): Encoder(
    (embedding): TransformerEmbedding(
      (token_embedding): Embedding(32000, 768, padding_idx=0)
      (position_embedding): Embedding(1024, 768)
      (dropout): Dropout(p=0.05, inplace=False)
    )
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (self_attn): MultiHeadAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (dropout1): Dropout(p=0.05, inplace=False)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (activation): GELU(approximate='none')
          (dr

In [6]:
# ============================================================
# CELL 6: GENERATION (KV-cache, batched, repetition control, forced length)
# ============================================================
def _apply_repetition_penalty(logits, gen, penalty):
    if penalty == 1.0 or gen.size(1) == 0:
        return logits
    score = logits.gather(1, gen)
    score = torch.where(score < 0, score * penalty, score / penalty)
    return logits.scatter(1, gen, score)

def _sample(logits, temperature, top_k, top_p):
    if temperature <= 0:
        return logits.argmax(dim=-1)
    probs = F.softmax(logits / temperature, dim=-1)
    if top_k and top_k > 0:
        kth = torch.topk(probs, top_k, dim=-1).values[:, -1:]
        probs = torch.where(probs < kth, torch.zeros_like(probs), probs)
    if top_p < 1.0:
        sp, si = torch.sort(probs, dim=-1, descending=True)
        remove = (torch.cumsum(sp, dim=-1) - sp) > top_p
        sp = sp.masked_fill(remove, 0.0)
        probs = torch.zeros_like(probs).scatter(1, si, sp)
    probs = probs / probs.sum(dim=-1, keepdim=True)
    return torch.multinomial(probs, 1).squeeze(1)

@torch.no_grad()
def generate_batch(m, src_lists, max_new_tokens=230, min_new_tokens=150, temperature=0.0, top_k=0,
                   top_p=1.0, rep_penalty=1.0, no_repeat_ngram=0, seed=0):
    m.eval()
    c = m.c
    assert max_new_tokens < c.dec_max
    torch.manual_seed(seed)
    B = len(src_lists); L = max(len(s) for s in src_lists)
    assert L <= c.enc_max, f"prompt has {L} tokens > encoder limit {c.enc_max}"
    src = torch.full((B, L), c.pad, dtype=torch.long)
    for i, s in enumerate(src_lists):
        src[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    src = src.to(DEVICE)
    with torch.autocast("cuda", dtype=torch.float16):
        src_mask, enc_kv = m.start_generation(src)
        caches = [None] * len(m.decoder.layers)
        tok = torch.full((B, 1), c.bos, dtype=torch.long, device=DEVICE)
        gen = torch.zeros((B, 0), dtype=torch.long, device=DEVICE)
        finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        hist = [[] for _ in range(B)]; maps = [dict() for _ in range(B)]
        for t in range(max_new_tokens):
            logits, caches = m.step(tok, t, caches, enc_kv, src_mask)
            logits[:, c.pad] = -float("inf"); logits[:, c.bos] = -float("inf"); logits[:, c.unk] = -float("inf")
            if t < min_new_tokens:
                logits[:, c.eos] = -float("inf")
            logits = _apply_repetition_penalty(logits, gen, rep_penalty)
            if no_repeat_ngram >= 2:
                n = no_repeat_ngram
                for b in range(B):
                    if len(hist[b]) >= n - 1:
                        banned = maps[b].get(tuple(hist[b][-(n - 1):]))
                        if banned:
                            logits[b, list(banned)] = -float("inf")
            nxt = _sample(logits, temperature, top_k, top_p)
            nxt = torch.where(finished, torch.full_like(nxt, c.pad), nxt)
            gen = torch.cat([gen, nxt[:, None]], dim=1)
            finished = finished | (nxt == c.eos)
            if no_repeat_ngram >= 2:
                for b, tk in enumerate(nxt.tolist()):
                    if tk == c.pad or tk == c.eos:
                        continue
                    hist[b].append(tk)
                    if len(hist[b]) >= no_repeat_ngram:
                        maps[b].setdefault(tuple(hist[b][-no_repeat_ngram:-1]), set()).add(tk)
            if bool(finished.all()):
                break
            tok = nxt[:, None]
    out = []
    for row in gen.tolist():
        ids = []
        for x in row:
            if x == c.eos or x == c.pad:
                break
            ids.append(x)
        out.append(ids)
    return out

def ids_to_text(ids, words=128):
    return " ".join(tokenizer.decode(ids).split()[:words])

def generate_texts(m, src_lists, batch_size=32, **kw):
    """Sort by length for efficiency, return texts in the original order."""
    order = sorted(range(len(src_lists)), key=lambda i: len(src_lists[i]))
    res = [None] * len(src_lists)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        outs = generate_batch(m, [src_lists[i] for i in idx], **kw)
        for i, o in zip(idx, outs):
            res[i] = ids_to_text(o)
    return res

# quick sanity check on 1 validation prompt
_src = [va_enc[0][va_enc[0] != 0].astype(int).tolist()]
print(generate_texts(model, _src, temperature=0.0, no_repeat_ngram=4)[0][:600])

the Market Committee shall be the Chairman of the Board. 4. [Omitted] [The words 'the Board' were deleted by G. S. R. 12 dated 12.9.1993.] (1) The Chairman shall be the Vice-Chairman of the Board and shall be the Secretary of the Board of Directors of the Board, and shall be responsible for the conduct of the business of the Board; (2) The Chairman may, by order in writing, delegate to the Chairman any of the powers and functions of the Board or any other officer of the Board under the Act; (3) The Chairman of the Committee shall be a member of the Board who shall be the Member-Secretary of th


In [7]:
# ============================================================
# CELL 7: CHOOSE THE DECODING SETTING ON VALIDATION PROMPTS
# ============================================================
sweep_idx = np.linspace(0, len(va_enc) - 1, SWEEP_N).astype(int)
sweep_src = [va_enc[i][va_enc[i] != 0].astype(int).tolist() for i in sweep_idx]
sweep_ref = []
for i in sweep_idx:
    ids = [int(x) for x in va_lab[i] if x != -100 and x != cfg.eos]
    sweep_ref.append(ids_to_text(ids))

def rouge_l_f(pred, ref):
    a, b = pred.split(), ref.split()
    if not a or not b: return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    l = prev[-1]
    if l == 0: return 0.0
    p, r = l / len(a), l / len(b)
    return 2 * p * r / (p + r)

def rep4(text):
    w = text.split(); g = [tuple(w[i:i + 4]) for i in range(len(w) - 3)]
    return 0.0 if not g else 1 - len(set(g)) / len(g)

def bertscore_f1(preds, refs):
    try:
        try:
            from bert_score import score
        except ImportError:
            os.system("pip install -q bert_score"); from bert_score import score
        _, _, f = score(preds, refs, lang="en", model_type="roberta-large", batch_size=16, device="cuda:0", verbose=False)
        torch.cuda.empty_cache()
        return float(f.mean())
    except Exception as e:
        print("   (BERTScore unavailable:", repr(e)[:120], ")"); return None

SWEEP = {
    "greedy":                  dict(temperature=0.0),
    "greedy_ng3":              dict(temperature=0.0, no_repeat_ngram=3),
    "greedy_ng4_rp1.15":       dict(temperature=0.0, no_repeat_ngram=4, rep_penalty=1.15),
    "top-p0.9_T0.7_ng4":       dict(temperature=0.7, top_p=0.9, no_repeat_ngram=4),
    "top-p0.9_T0.5_ng4_rp1.1": dict(temperature=0.5, top_p=0.9, no_repeat_ngram=4, rep_penalty=1.1),
    "top-k40_T0.6_ng4":        dict(temperature=0.6, top_k=40, no_repeat_ngram=4),
}
if SMOKE:
    SWEEP = {k: SWEEP[k] for k in ("greedy", "greedy_ng4_rp1.15")}

sweep_results = {}
for name, kw in SWEEP.items():
    t0 = time.time()
    preds = generate_texts(model, sweep_src, **kw)
    rl = float(np.mean([rouge_l_f(p, r) for p, r in zip(preds, sweep_ref)]))
    rp = float(np.mean([rep4(p) for p in preds]))
    wd = float(np.mean([len(p.split()) for p in preds]))
    bs = bertscore_f1(preds, sweep_ref)
    sweep_results[name] = dict(rougeL=rl, rep4=rp, words=wd, bertscore=bs)
    print(f"{name:26s} ROUGE-L {rl:.4f} | rep4 {rp:.3f} | words {wd:5.1f} | BERTScore {bs if bs is None else round(bs,4)} | {time.time()-t0:.0f}s")

key = (lambda n: sweep_results[n]["bertscore"]) if all(v["bertscore"] is not None for v in sweep_results.values()) \
      else (lambda n: sweep_results[n]["rougeL"])
BEST_NAME = max(sweep_results, key=key)
BEST_DECODE = SWEEP[BEST_NAME]
print("\nSELECTED DECODING:", BEST_NAME, BEST_DECODE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (16 outputs left out)

top-p0.9_T0.5_ng4_rp1.1    ROUGE-L 0.1923 | rep4 0.001 | words 127.3 | BERTScore 0.8349 | 30s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


top-k40_T0.6_ng4           ROUGE-L 0.1921 | rep4 0.001 | words 127.3 | BERTScore 0.8298 | 28s

SELECTED DECODING: top-p0.9_T0.5_ng4_rp1.1 {'temperature': 0.5, 'top_p': 0.9, 'no_repeat_ngram': 4, 'rep_penalty': 1.1}


In [8]:
# ============================================================
# CELL 8: OFFICIAL TEST PREDICTIONS -> prediction.json + submission.zip
# ============================================================
import requests
if TEST_JSON_PATH:
    test_data = json.load(open(TEST_JSON_PATH))
else:
    headers = {}
    try:
        from kaggle_secrets import UserSecretsClient
        headers = {"Authorization": "Bearer " + UserSecretsClient().get_secret("HF")}
    except Exception as e:
        print("No HF secret found, trying without token:", repr(e)[:80])
    url = "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Test/test.json"
    r = requests.get(url, headers=headers, timeout=180); r.raise_for_status()
    test_data = r.json()

ids_all = list(test_data.keys())
ids_run = ids_all[:TEST_LIMIT] if TEST_LIMIT else ids_all
test_src = []
for k in ids_run:
    text = test_data[k]["text"]
    if len(text.split()) != 512:
        print(f"[note] prompt {k} has {len(text.split())} words (expected 512)")
    test_src.append(tokenizer.encode(text).ids)
print("test prompts:", len(ids_run), "| max prompt tokens:", max(len(s) for s in test_src))

t0 = time.time()
texts = generate_texts(model, test_src, **BEST_DECODE)
predictions = {k: {"text": t} for k, t in zip(ids_run, texts)}
wc = [len(v["text"].split()) for v in predictions.values()]
print(f"generated in {(time.time()-t0)/60:.1f} min | words min/avg/max = {min(wc)}/{np.mean(wc):.1f}/{max(wc)} | exactly 128: {sum(w==128 for w in wc)}")

if SMOKE:
    json.dump(predictions, open("/kaggle/working/smoke_prediction.json", "w"), indent=1)
    print("SMOKE run only: wrote smoke_prediction.json (NOT for submission)")
else:
    assert set(predictions) == set(ids_all) and all("text" in v for v in predictions.values())
    json.dump(predictions, open("/kaggle/working/prediction.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write("/kaggle/working/prediction.json", arcname="prediction.json")
    print("zip contents:", zipfile.ZipFile("/kaggle/working/submission.zip").namelist())

summary = dict(preset=PRESET, params=n_params, config=cfg.__dict__, trained=TRAIN, time_budget_h=TIME_BUDGET_HOURS,
               best_val_nll=(train_state["best_val"] if TRAIN else None), history=(train_state["history"] if TRAIN else None),
               decode_sweep=sweep_results, selected_decode=BEST_NAME, selected_decode_kwargs=BEST_DECODE,
               lr=PEAK_LR, eff_batch=BATCH_PER_FORWARD * ACCUM, data_dir=DATA_DIR)
json.dump(summary, open(os.path.join(OUT_DIR, "experiment_summary.json"), "w"), indent=1, default=str)
print("saved experiment_summary.json (use it for the report)")

test prompts: 1000 | max prompt tokens: 994
generated in 1.1 min | words min/avg/max = 87/127.6/128 | exactly 128: 795
zip contents: ['prediction.json']
saved experiment_summary.json (use it for the report)
